In [17]:
import requests
import json
import time

API_URL = "https://www.euroncap.com/api/CarListRoute"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
}
PARAMS = {
    "path": "/assessments",
    "limit": 20,
    "offset": 0,
    "sort": "protocol.protocolYear:desc,make.name:asc",
}

all_items = []
print("Fetching Euroncap assessment list...")

while True:
    response = requests.get(API_URL, headers=HEADERS, params=PARAMS, timeout=30)
    response.raise_for_status()
    page = response.json()

    items = page.get("items", [])
    if not items:
        print("No items returned. Stopping.")
        break

    all_items.extend(items)
    meta = page.get("meta", {})
    total = meta.get("totalCount", len(all_items))
    print(f"Fetched {len(items)} items (offset {PARAMS['offset']}), total expected {total}.")

    if len(all_items) >= total:
        break

    PARAMS["offset"] += PARAMS["limit"]
    time.sleep(0.5)

vehicle_links = []
for item in all_items:
    assessment_id = item.get("id")
    if not assessment_id:
        continue
    link = f"https://www.euroncap.com/assessments/?assessmentId={assessment_id}"
    make = item.get("make", {}).get("name")
    model = item.get("model", {}).get("name")
    year = item.get("publicationYear") or item.get("ratingYear")
    vehicle_links.append({
        "assessmentId": assessment_id,
        "make": make,
        "model": model,
        "year": year,
        "assessmentUrl": link,
    })

output_file = "vehicle_links.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(vehicle_links, f, indent=2, ensure_ascii=False)

print(f"Saved {len(vehicle_links)} vehicle link entries to {output_file}.")
print("Sample links:")
for item in vehicle_links[:10]:
    print(item)

Fetching Euroncap assessment list...
Fetched 20 items (offset 0), total expected 570.
Fetched 20 items (offset 20), total expected 570.
Fetched 20 items (offset 40), total expected 570.
Fetched 20 items (offset 60), total expected 570.
Fetched 20 items (offset 80), total expected 570.
Fetched 20 items (offset 100), total expected 570.
Fetched 20 items (offset 120), total expected 570.
Fetched 20 items (offset 140), total expected 570.
Fetched 20 items (offset 160), total expected 570.
Fetched 20 items (offset 180), total expected 570.
Fetched 20 items (offset 200), total expected 570.
Fetched 20 items (offset 220), total expected 570.
Fetched 20 items (offset 240), total expected 570.
Fetched 20 items (offset 260), total expected 570.
Fetched 20 items (offset 280), total expected 570.
Fetched 20 items (offset 300), total expected 570.
Fetched 20 items (offset 320), total expected 570.
Fetched 20 items (offset 340), total expected 570.
Fetched 20 items (offset 360), total expected 570.


In [18]:
from pathlib import Path

base_url = 'https://www.euroncap.com/api/CarListRoute?path=%2Fassessments%2F'
output_folder = Path('test_data')
output_folder.mkdir(parents=True, exist_ok=True)

for item in vehicle_links:
    assessment_id = item.get('assessmentId')
    if not assessment_id:
        continue

    json_url = base_url + str(assessment_id)
    response = requests.get(json_url, headers=HEADERS, timeout=30)
    response.raise_for_status()

    file_path = output_folder / f"{assessment_id}.json"
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write(response.text)

    print(f"Downloaded {assessment_id} -> {file_path}")

Downloaded h032 -> test_data/h032.json
Downloaded h039 -> test_data/h039.json
Downloaded h031 -> test_data/h031.json
Downloaded h025 -> test_data/h025.json
Downloaded h034 -> test_data/h034.json
Downloaded h035 -> test_data/h035.json
Downloaded h036 -> test_data/h036.json
Downloaded h054 -> test_data/h054.json
Downloaded h043 -> test_data/h043.json
Downloaded h037 -> test_data/h037.json
Downloaded h042 -> test_data/h042.json
Downloaded h050 -> test_data/h050.json
Downloaded h051 -> test_data/h051.json
Downloaded h038 -> test_data/h038.json
Downloaded 1169 -> test_data/1169.json
Downloaded 1130 -> test_data/1130.json
Downloaded 1182 -> test_data/1182.json
Downloaded 1106 -> test_data/1106.json
Downloaded 1149 -> test_data/1149.json
Downloaded 1138 -> test_data/1138.json
Downloaded 1138 -> test_data/1138.json
Downloaded 1126 -> test_data/1126.json
Downloaded 1146 -> test_data/1146.json
Downloaded 1125 -> test_data/1125.json
Downloaded a032 -> test_data/a032.json
Downloaded 1210 -> test_d

In [2]:
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 24.4 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 39.2 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [ ]:
import json
import sqlite3
import pandas as pd
import os
from pathlib import Path

# Mapping NCAP injury colors to numeric values for ML
# 4 is the safest (Green), 0 is the most dangerous (Red)
COLOR_MAP = {
    "GREEN": 4,
    "YELLOW": 3,
    "ORANGE": 2,
    "BROWN": 1,
    "RED": 0
}

def safe_get(data, *keys):
    """Safely navigates nested dictionaries, returning None if any key is missing."""
    for key in keys:
        if isinstance(data, dict):
            data = data.get(key)
        else:
            return None
    return data

def get_injury_val(dummy_dict, seat=None, body_part=None):
    """Helper to convert color text to numeric score.

    Supports both:
      get_injury_val(dummy_dict, "head")
      get_injury_val(dummy_dict, "seat1", "head")
    """
    if body_part is None:
        body_part = seat
        seat = None

    if seat is None:
        color = safe_get(dummy_dict, body_part)
    else:
        color = safe_get(dummy_dict, seat, body_part)
    return COLOR_MAP.get(color)  # Returns None if color is not in map or is None

def extract_important_data(json_file_path):
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # 1. METADATA & IDENTITY
    a_type = safe_get(data, 'assessmentType', 'code')
    features = {
        "id": data.get("id"),
        "make": safe_get(data, "make", "name"),
        "model": safe_get(data, "model", "name"),
        "year": data.get("ratingYear"),
        "type": a_type,
        "stars": (safe_get(data, "carSafety", "safetyRatingStars") or 
                  safe_get(data, "vanSafety", "safetyRatingStars")),
        "drive_hand": safe_get(data, "testedHandOfDrive", "code"),
    }

    # 2. CAR SAFETY SPECIFIC DATA
    if a_type == "CAR_SAFETY":
        a_data = safe_get(data, 'assessmentData', 'carSafety') or {}
        perf = a_data.get('safetyPerformance', {})
        equip = a_data.get('safetyEquipment', {})
        
        # Physical & Core Scores
        features.update({
            "kerb_weight": perf.get("kerbWeight"),
            "adult_occupant_score": safe_get(perf, "adultOccupant", "normalisedScore"),
            "child_occupant_score": safe_get(perf, "childOccupant", "normalisedScore"),
            "vru_score": safe_get(perf, "vulnerableRoadUsers", "normalisedScore"),
            "safety_assist_score": safe_get(perf, "safetyAssist", "normalisedScore"),
            "adult_frontal_impact_raw": safe_get(perf, "adultOccupant", "frontalImpact", "score"),
            "adult_side_impact_raw": safe_get(perf, "adultOccupant", "sideImpact", "score"),
        })

        # Safety Equipment Booleans (1 if Fitted Standard, else 0)
        features.update({
            "has_active_bonnet": 1 if safe_get(equip, "otherSystems", "activeBonnet") == "FITTED_STANDARD" else 0,
            "has_lane_assist": 1 if safe_get(equip, "otherSystems", "laneAssistSystem") == "FITTED_STANDARD" else 0,
            "has_speed_assist": 1 if safe_get(equip, "otherSystems", "speedAssistance") == "FITTED_STANDARD" else 0,
            "has_aeb_car": 1 if safe_get(equip, "otherSystems", "AEBCarToCar") == "FITTED_STANDARD" else 0,
        })

        # --- INJURY COLORS (Converted to Numbers) ---
        # Frontal Offset - Driver (Seat 1)
        off_dr = safe_get(perf, "adultOccupant", "frontalImpact", "offset", "dummyVisual", "seat1")
        features.update({
            "inj_off_dr_head": get_injury_val(off_dr, "head"),
            "inj_off_dr_chest": get_injury_val(off_dr, "chest"),
            "inj_off_dr_thigh": get_injury_val(off_dr, "leftFemur"),
        })

        # Frontal Full Width - Rear Passenger (Seat 6)
        fw_rear = safe_get(perf, "adultOccupant", "frontalImpact", "FW", "dummyVisual", "seat6")
        features.update({
            "inj_fw_rear_head": get_injury_val(fw_rear, "head"),
            "inj_fw_rear_chest": get_injury_val(fw_rear, "chest"),
            "inj_fw_rear_pelvis": get_injury_val(fw_rear, "pelvis"),
        })

        # Side Impact (MDB) - Driver (Seat 1)
        side_mdb = safe_get(perf, "adultOccupant", "sideImpact", "MDB", "dummyVisual", "seat1")
        features.update({
            "inj_side_dr_head": get_injury_val(side_mdb, "head"),
            "inj_side_dr_chest": get_injury_val(side_mdb, "chest"),
            "inj_side_dr_pelvis": get_injury_val(side_mdb, "pelvis"),
        })

    # 3. VAN SAFETY SPECIFIC DATA (To prevent errors and include van weight)
    elif a_type == "VAN_SAFETY":
        v_data = safe_get(data, 'assessmentData', 'vanSafety', 'safetyPerformance') or {}
        variant = safe_get(data, 'variants', 0, 'genericVariantData')
        
        features.update({
            "kerb_weight": safe_get(variant, 'kerbWeight', 'value'),
            "safety_assist_score": v_data.get("crashAvoidance", {}).get("normalisedScore"),
            # Vans don't have crash injury color data in the same format, so these remain None
        })

    return features

def process_directory_to_db(json_folder, db_name="ncap_ml_ready.db"):
    all_vehicles = []
    pathlist = Path(json_folder).glob('*.json')
    
    for path in pathlist:
        try:
            vehicle_data = extract_important_data(str(path))
            all_vehicles.append(vehicle_data)
        except Exception as e:
            print(f"Error processing {path.name}: {e}")

    df = pd.DataFrame(all_vehicles)
    
    # Save to SQLite for your database requirement
    conn = sqlite3.connect(db_name)
    df.to_sql('vehicles', conn, if_exists='replace', index=False)
    conn.close()
    
    print(f"Success! Processed {len(df)} vehicles.")
    return df

# --- EXECUTION ---
# Make sure your folder 'test_data/' exists and contains your JSONs
df = process_directory_to_db('test_data/')

# Show a glimpse of the key columns
print(df[['make', 'model', 'stars', 'inj_off_dr_head', 'inj_fw_rear_chest', 'has_aeb_car']].head())

Error processing test_data/v057.json: 'NoneType' object has no attribute 'get'
Error processing test_data/h019.json: 'NoneType' object has no attribute 'get'
Error processing test_data/h023.json: 'NoneType' object has no attribute 'get'
Error processing test_data/h018.json: 'NoneType' object has no attribute 'get'
Error processing test_data/h024.json: 'NoneType' object has no attribute 'get'
Error processing test_data/v107.json: 'NoneType' object has no attribute 'get'
Error processing test_data/h020.json: 'NoneType' object has no attribute 'get'
Error processing test_data/v086.json: 'NoneType' object has no attribute 'get'
Error processing test_data/v092.json: 'NoneType' object has no attribute 'get'
Error processing test_data/a025.json: 'NoneType' object has no attribute 'get'
Error processing test_data/v090.json: 'NoneType' object has no attribute 'get'
Error processing test_data/h014.json: 'NoneType' object has no attribute 'get'
Error processing test_data/h047.json: 'NoneType' obj

In [1]:
import json
import sqlite3
import pandas as pd
import os
from pathlib import Path

# Mapping NCAP injury colors to numeric values for ML
# 4 is the safest (Green), 0 is the most dangerous (Red)
COLOR_MAP = {
    "GREEN": 4,
    "YELLOW": 3,
    "ORANGE": 2,
    "BROWN": 1,
    "RED": 0
}

def safe_get(data, *keys):
    """Safely navigates nested dictionaries, returning None if any key is missing."""
    for key in keys:
        if isinstance(data, dict):
            data = data.get(key)
        else:
            return None
    return data

def get_injury_val(dummy_dict, seat=None, body_part=None):
    """Helper to convert color text to numeric score.

    Supports both:
      get_injury_val(dummy_dict, "head")
      get_injury_val(dummy_dict, "seat1", "head")
    """
    if body_part is None:
        body_part = seat
        seat = None

    if seat is None:
        color = safe_get(dummy_dict, body_part)
    else:
        color = safe_get(dummy_dict, seat, body_part)
    return COLOR_MAP.get(color)  # Returns None if color is not in map or is None

def is_std(val):
    """Helper to check if equipment is standard (returns 1) or not (returns 0)."""
    return 1 if val == "FITTED_STANDARD" else 0

def extract_important_data(json_file_path):
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # 1. METADATA & IDENTITY
    a_type = safe_get(data, 'assessmentType', 'code')
    features = {
        "id": data.get("id"),
        "make": safe_get(data, "make", "name"),
        "model": safe_get(data, "model", "name"),
        "year": data.get("ratingYear"),
        "type": a_type,
        "stars": (safe_get(data, "carSafety", "safetyRatingStars") or 
                  safe_get(data, "vanSafety", "safetyRatingStars")),
        "drive_hand": safe_get(data, "testedHandOfDrive", "code"),
        "protocol_year": safe_get(data, "protocol", "protocolYear"),
    }

    # 2. CAR SAFETY SPECIFIC DATA
    if a_type == "CAR_SAFETY":
        a_data = safe_get(data, 'assessmentData', 'carSafety') or {}
        perf = a_data.get('safetyPerformance', {})
        equip = a_data.get('safetyEquipment', {})
        
        # Physical & Core Scores
        features.update({
            "kerb_weight": perf.get("kerbWeight"),
            'seat_arrangement': perf.get("seatArrangement"),
            'rating_year': perf.get("ratingYear"),
            "adult_occupant_score": safe_get(perf, "adultOccupant", "normalisedScore"),
            "child_occupant_score": safe_get(perf, "childOccupant", "normalisedScore"),
            "vru_score": safe_get(perf, "vulnerableRoadUsers", "normalisedScore"),
            "safety_assist_score": safe_get(perf, "safetyAssist", "normalisedScore"),
            "adult_frontal_impact_raw": safe_get(perf, "adultOccupant", "frontalImpact", "score"),
            "adult_side_impact_raw": safe_get(perf, "adultOccupant", "sideImpact", "score"),
            "adult_rear_impact_raw": safe_get(perf, "adultOccupant", "rearImpact", "score"),
            "adult_rescue_extrication_raw": safe_get(perf, "adultOccupant", "rescueAndExtrication", "score"),
            "child_cop_dynamic_raw": safe_get(perf, "childOccupant", "COPDynamic", "score"),
        })

        # --- SAFETY EQUIPMENT ---
        features.update({
            # Frontal Airbags
            "seat1_frontal_airbag": is_std(safe_get(equip, "frontalCrashProtection", "frontalAirbag", "seat1")),
            "seat3_frontal_airbag": is_std(safe_get(equip, "frontalCrashProtection", "frontalAirbag", "seat3")),
            "row2_frontal_airbag": is_std(safe_get(equip, "frontalCrashProtection", "frontalAirbag", "row2")),
            # Belt Pretensioners
            "seat1_pretensioner": is_std(safe_get(equip, "frontalCrashProtection", "beltPretensioner", "seat1")),
            "seat3_pretensioner": is_std(safe_get(equip, "frontalCrashProtection", "beltPretensioner", "seat3")),
            "row2_pretensioner": is_std(safe_get(equip, "frontalCrashProtection", "beltPretensioner", "row2")),
            # Belt Load Limiters
            "seat1_load_limiter": is_std(safe_get(equip, "frontalCrashProtection", "beltLoadLimiter", "seat1")),
            "seat3_load_limiter": is_std(safe_get(equip, "frontalCrashProtection", "beltLoadLimiter", "seat3")),
            "row2_load_limiter": is_std(safe_get(equip, "frontalCrashProtection", "beltLoadLimiter", "row2")),
            # Knee Airbags
            "seat1_knee_airbag": is_std(safe_get(equip, "frontalCrashProtection", "kneeAirbag", "seat1")),
            "seat3_knee_airbag": is_std(safe_get(equip, "frontalCrashProtection", "kneeAirbag", "seat3")),
            # Lateral / Side Head Airbags
            "seat1_side_head_airbag": is_std(safe_get(equip, "lateralCrashProtection", "sideHeadAirbag", "seat1")),
            "seat3_side_head_airbag": is_std(safe_get(equip, "lateralCrashProtection", "sideHeadAirbag", "seat3")),
            "row2_side_head_airbag": is_std(safe_get(equip, "lateralCrashProtection", "sideHeadAirbag", "row2")),
            # Side Chest Airbags
            "seat1_side_chest_airbag": is_std(safe_get(equip, "lateralCrashProtection", "sideChestAirbag", "seat1")),
            "seat3_side_chest_airbag": is_std(safe_get(equip, "lateralCrashProtection", "sideChestAirbag", "seat3")),
            "row2_side_chest_airbag": is_std(safe_get(equip, "lateralCrashProtection", "sideChestAirbag", "row2")),
            # Side Pelvis Airbags
            "seat1_side_pelvis_airbag": is_std(safe_get(equip, "lateralCrashProtection", "sidePelvisAirbag", "seat1")),
            "seat3_side_pelvis_airbag": is_std(safe_get(equip, "lateralCrashProtection", "sidePelvisAirbag", "seat3")),
            "row2_side_pelvis_airbag": is_std(safe_get(equip, "lateralCrashProtection", "sidePelvisAirbag", "row2")),
            # Centre Airbags for lateral
            "seat1_centre_lateral_airbag": is_std(safe_get(equip, "lateralCrashProtection", "centreAirbag", "seat1")),
            "seat3_centre_lateral_airbag": is_std(safe_get(equip, "lateralCrashProtection", "centreAirbag", "seat3")),
            "row2_centre_lateral_airbag": is_std(safe_get(equip, "lateralCrashProtection", "centreAirbag", "row2")),
            # Child Protection
            "seat1_isofix": is_std(safe_get(equip, "childProtection", "isofix", "seat1")),
            "seat3_isofix": is_std(safe_get(equip, "childProtection", "isofix", "seat3")),
            "row2_isofix": is_std(safe_get(equip, "childProtection", "isofix", "row2")),
            "row3_isofix": is_std(safe_get(equip, "childProtection", "isofix", "row3")),

            "seat1_isize": is_std(safe_get(equip, "childProtection", "iSize", "seat1")),
            "seat3_isize": is_std(safe_get(equip, "childProtection", "iSize", "seat3")),
            "row2_isize": is_std(safe_get(equip, "childProtection", "iSize", "row2")),
            "row3_isize": is_std(safe_get(equip, "childProtection", "iSize", "row3")),

            "seat1_integrated_child_seat": is_std(safe_get(equip, "childProtection", "integratedChildSeat", "seat1")),
            "seat3_integrated_child_seat": is_std(safe_get(equip, "childProtection", "integratedChildSeat", "seat3")),
            "row2_integrated_child_seat": is_std(safe_get(equip, "childProtection", "integratedChildSeat", "row2")),
            "row3_integrated_child_seat": is_std(safe_get(equip, "childProtection", "integratedChildSeat", "row3")),
            
            "seat1_airbagcutoffswitch": is_std(safe_get(equip, "childProtection", "airbagCutoffSwitch", "seat1")),
            "seat3_airbagcutoffswitch": is_std(safe_get(equip, "childProtection", "airbagCutoffSwitch", "seat3")),
            "row2_airbagcutoffswitch": is_std(safe_get(equip, "childProtection", "airbagCutoffSwitch", "row2")),
            "row3_airbagcutoffswitch": is_std(safe_get(equip, "childProtection", "airbagCutoffSwitch", "row3")),

            "seat1_childdetection": is_std(safe_get(equip, "childProtection", "childPresenceDetection", "seat1")),
            "seat3_childdetection": is_std(safe_get(equip, "childProtection", "childPresenceDetection", "seat3")),
            "row2_childdetection": is_std(safe_get(equip, "childProtection", "childPresenceDetection", "row2")),
            "row3_childdetection": is_std(safe_get(equip, "childProtection", "childPresenceDetection", "row3")),

            # Other Systems (ADAD)
            "has_active_bonnet": is_std(safe_get(equip, "otherSystems", "activeBonnet")),
            "has_aeb_vru": is_std(safe_get(equip, "otherSystems", "AEBVulnerableRoadUsers")),
            "has_cyclistdoorprevention": is_std(safe_get(equip, "otherSystems", "cyclistDooringPrevention")),
            "has_aeb_m2w": is_std(safe_get(equip, "otherSystems", "AEBMotorcyclist")),
            "has_aeb_cartocar": is_std(safe_get(equip, "otherSystems", "AEBCarToCar")),
            "has_speed_assist": is_std(safe_get(equip, "otherSystems", "speedAssistance")),
            "has_lane_assist": is_std(safe_get(equip, "otherSystems", "laneAssistSystem")),
            "has_fatigue_detection": is_std(safe_get(equip, "otherSystems", "fatigueDetection")),
            "has_distraction_detection": is_std(safe_get(equip, "otherSystems", "distractionDetection")),
        })

        # --- INJURY COLORS (ADULT) ---
        off_dr = safe_get(perf, "adultOccupant", "frontalImpact", "offset", "dummyVisual")
        features.update({
            'fr_offset_score': safe_get(perf, "adultOccupant", "frontalImpact", "offset", "score"),
            "seat1_fr_offset_dummy_type": safe_get(off_dr, 'seat1', 'dummyType'),
            "seat1_fr_offset_head_injury": get_injury_val(off_dr,"seat1", "head"),
            "seat1_fr_offset_neck_injury": get_injury_val(off_dr,"seat1", "neck"),
            "seat1_fr_offset_chest_injury": get_injury_val(off_dr,"seat1", "chest"),
            "seat1_fr_offset_abdomen_injury": get_injury_val(off_dr,"seat1", "abdomen"),
            "seat1_fr_offset_pelvis_injury": get_injury_val(off_dr,"seat1", "pelvis"),
            "seat1_fr_offset_left_Femur_injury": get_injury_val(off_dr,"seat1", "leftFemur"),
            "seat1_fr_offset_right_Femur_injury": get_injury_val(off_dr,"seat1", "rightFemur"),
            "seat1_fr_offset_left_Tibia_injury": get_injury_val(off_dr,"seat1", "leftTibia"),
            "seat1_fr_offset_right_Tibia_injury": get_injury_val(off_dr,"seat1", "rightTibia"),
            "seat3_fr_offset_dummy_type": safe_get(off_dr, 'seat3', 'dummyType'),
            "seat3_fr_offset_head_injury": get_injury_val(off_dr,"seat3", "head"),
            "seat3_fr_offset_neck_injury": get_injury_val(off_dr,"seat3", "neck"),
            "seat3_fr_offset_chest_injury": get_injury_val(off_dr,"seat3", "chest"),
            "seat3_fr_offset_left_Femur_injury": get_injury_val(off_dr,"seat3", "leftFemur"),
            "seat3_fr_offset_right_Femur_injury": get_injury_val(off_dr,"seat3", "rightFemur"),
            "seat3_fr_offset_left_Tibia_injury": get_injury_val(off_dr,"seat3", "leftTibia"),
            "seat3_fr_offset_right_Tibia_injury": get_injury_val(off_dr,"seat3", "rightTibia"),
        })

        # -------------------------------------------------------------------------------- 
        # --------------------------------------------------------------------------------
        # I need to start with fw injuries and cop values are also left
        # --------------------------------------------------------------------------------
        # --------------------------------------------------------------------------------
        
        fw_rear = safe_get(perf, "adultOccupant", "frontalImpact", "FW", "dummyVisual")
        features.update({
            'fr_fw_score': safe_get(perf, "adultOccupant", "frontalImpact", "FW", "score"),
            "seat1_fr_fw_dummy_type": safe_get(fw_rear, 'seat1', 'dummyType'),
            "seat1_fr_fw_head_injury": get_injury_val(fw_rear, 'seat1', "head"),
            "seat1_fr_fw_neck_injury": get_injury_val(fw_rear, 'seat1', "neck"),
            "seat1_fr_fw_chest_injury": get_injury_val(fw_rear, 'seat1', "chest"),
            "seat1_fr_fw_pelvis_injury": get_injury_val(fw_rear, 'seat1', "pelvis"),
            "seat1_fr_fw_left_Femur_injury": get_injury_val(fw_rear, 'seat1', "leftFemur"),
            "seat1_fr_fw_right_Femur_injury": get_injury_val(fw_rear, 'seat1', "rightFemur"),
            "seat6_fr_fw_dummy_type": safe_get(fw_rear, 'seat6', 'dummyType'),
            "seat6_fr_fw_head_injury": get_injury_val(fw_rear, 'seat6', "head"),
            "seat6_fr_fw_neck_injury": get_injury_val(fw_rear, 'seat6', "neck"),
            "seat6_fr_fw_chest_injury": get_injury_val(fw_rear, 'seat6', "chest"),
            "seat6_fr_fw_pelvis_injury": get_injury_val(fw_rear, 'seat6', "pelvis"),
            "seat6_fr_fw_left_Femur_injury": get_injury_val(fw_rear, 'seat6', "leftFemur"),
            "seat6_fr_fw_right_Femur_injury": get_injury_val(fw_rear, 'seat6', "rightFemur")
        })

        side_mdb = safe_get(perf, "adultOccupant", "sideImpact", "MDB", "dummyVisual")
        side_pole = safe_get(perf, "adultOccupant", "sideImpact", "sidePole", "dummyVisual")
        side_farside = safe_get(perf, "adultOccupant", "sideImpact", "sideFarside", "dummyVisual")
        features.update({
            'side_mdb_score': safe_get(perf, "adultOccupant", "sideImpact", "MDB", "score"),
            "seat1_side_mdb_dummy_type": safe_get(side_mdb, 'seat1', 'dummyType'),
            "seat1_side_mdb_head_injury": get_injury_val(side_mdb, "seat1", "head"),
            "seat1_side_mdb_chest_injury": get_injury_val(side_mdb, "seat1", "chest"),
            "seat1_side_mdb_abdomen_injury": get_injury_val(side_mdb, "seat1", "abdomen"),
            "seat1_side_mdb_pelvis_injury": get_injury_val(side_mdb, "seat1", "pelvis"),
            'side_pole_score': safe_get(perf, "adultOccupant", "sideImpact", "sidePole", "score"),
            "seat1_side_pole_dummy_type": safe_get(side_pole, 'seat1', 'dummyType'),
            "seat1_side_pole_head_injury": get_injury_val(side_pole, "seat1", "head"),
            "seat1_side_pole_chest_injury": get_injury_val(side_pole, "seat1", "chest"),
            "seat1_side_pole_abdomen_injury": get_injury_val(side_pole, "seat1", "abdomen"),
            "seat1_side_pole_pelvis_injury": get_injury_val(side_pole, "seat1", "pelvis"),
            "seat1_side_farside_dummy_type": safe_get(side_farside, 'seat1', 'dummyType'),
            "seat1_side_farside_excursion_zone": safe_get(side_farside, 'seat1', 'excursionZone'),
            "seat1_side_farside_head_injury": get_injury_val(side_farside, "seat1", "head"),
            "seat1_side_farside_neck_injury": get_injury_val(side_farside, "seat1", "neck"),
            "seat1_side_farside_chest_injury": get_injury_val(side_farside, "seat1", "chest"),
            "seat1_side_farside_pelvis_injury": get_injury_val(side_farside, "seat1", "pelvis"),
        })

        # --- INJURY COLORS (CHILD COP) ---
        # Frontal Impact Dummy Visuals (Seat 4 & Seat 6)
        cop_f_s4 = safe_get(perf, "childOccupant", "COPDynamic", "frontalImpact", "dummyVisual", "seat4")
        cop_f_s6 = safe_get(perf, "childOccupant", "COPDynamic", "frontalImpact", "dummyVisual", "seat6")
        features.update({
            "inj_child_f_s4_head": get_injury_val(cop_f_s4, "head"),
            "inj_child_f_s4_neck": get_injury_val(cop_f_s4, "neck"),
            "inj_child_f_s4_chest": get_injury_val(cop_f_s4, "chest"),
            "inj_child_f_s6_head": get_injury_val(cop_f_s6, "head"),
            "inj_child_f_s6_neck": get_injury_val(cop_f_s6, "neck"),
            "inj_child_f_s6_chest": get_injury_val(cop_f_s6, "chest"),
        })

        # Side Impact Dummy Visuals (Seat 4 & Seat 6)
        cop_s_s4 = safe_get(perf, "childOccupant", "COPDynamic", "sideImpact", "dummyVisual", "seat4")
        cop_s_s6 = safe_get(perf, "childOccupant", "COPDynamic", "sideImpact", "dummyVisual", "seat6")
        features.update({
            "inj_child_s_s4_head": get_injury_val(cop_s_s4, "head"),
            "inj_child_s_s4_neck": get_injury_val(cop_s_s4, "neck"),
            "inj_child_s_s4_chest": get_injury_val(cop_s_s4, "chest"),
            "inj_child_s_s6_head": get_injury_val(cop_s_s6, "head"),
            "inj_child_s_s6_neck": get_injury_val(cop_s_s6, "neck"),
            "inj_child_s_s6_chest": get_injury_val(cop_s_s6, "chest"),
        })

    # 3. VAN SAFETY SPECIFIC DATA
    elif a_type == "VAN_SAFETY":
        v_data = safe_get(data, 'assessmentData', 'vanSafety', 'safetyPerformance') or {}
        variant = safe_get(data, 'variants', 0, 'genericVariantData')
        
        features.update({
            "kerb_weight": safe_get(variant, 'kerbWeight', 'value'),
            "safety_assist_score": v_data.get("crashAvoidance", {}).get("normalisedScore"),
        })

    return features

def process_directory_to_db(json_folder, db_name="ncap_ml_ready.db"):
    all_vehicles = []
    pathlist = Path(json_folder).glob('*.json')
    
    for path in pathlist:
        try:
            vehicle_data = extract_important_data(str(path))
            all_vehicles.append(vehicle_data)
        except Exception as e:
            print(f"Error processing {path.name}: {e}")

    df = pd.DataFrame(all_vehicles)
    
    conn = sqlite3.connect(db_name)
    df.to_sql('vehicles', conn, if_exists='replace', index=False)
    conn.close()
    
    print(f"Success! Processed {len(df)} vehicles into {db_name}.")
    return df

# --- EXECUTION ---
df = process_directory_to_db('test_data/')
print(df.head())

Success! Processed 562 vehicles into ncap_ml_ready.db.
     id    make     model  year          type  stars drive_hand  \
0  v057   IVECO     Daily  2025    VAN_SAFETY    4.0        NaN   
1  h019  Scania  R-series  2025  TRUCK_SAFETY    NaN        NaN   
2  1171    Togg      T10F  2025    CAR_SAFETY    5.0        LHD   
3  0850  Toyota     Mirai  2021    CAR_SAFETY    5.0        LHD   
4  h023     MAN       TGX  2025  TRUCK_SAFETY    NaN        NaN   

   protocol_year  kerb_weight  safety_assist_score  ... inj_child_f_s4_chest  \
0           2025          NaN                 73.0  ...                  NaN   
1           2025          NaN                  NaN  ...                  NaN   
2           2025       2124.0                 80.0  ...                  4.0   
3           2021       1950.0                 82.0  ...                  4.0   
4           2025          NaN                  NaN  ...                  NaN   

   inj_child_f_s6_head  inj_child_f_s6_neck  inj_child_f_s6_c

In [2]:
df

,id,make,model,year,type,stars,drive_hand,protocol_year,kerb_weight,safety_assist_score,...,inj_child_f_s4_chest,inj_child_f_s6_head,inj_child_f_s6_neck,inj_child_f_s6_chest,inj_child_s_s4_head,inj_child_s_s4_neck,inj_child_s_s4_chest,inj_child_s_s6_head,inj_child_s_s6_neck,inj_child_s_s6_chest
0,v057,IVECO,Daily,2025,VAN_SAFETY,4.0,NaN,2025,NaN,73.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,h019,Scania,R-series,2025,TRUCK_SAFETY,NaN,NaN,2025,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1171,Togg,T10F,2025,CAR_SAFETY,5.0,LHD,2025,2124.0,80.0,...,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0
3,0850,Toyota,Mirai,2021,CAR_SAFETY,5.0,LHD,2021,1950.0,82.0,...,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0
4,h023,MAN,TGX,2025,TRUCK_SAFETY,NaN,NaN,2025,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
557,0737,Lexus,ES,2018,CAR_SAFETY,5.0,RHD,2018,1740.0,77.0,...,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0
558,1021,Volkswagen,Passat,2024,CAR_SAFETY,5.0,LHD,2024,1618.0,80.0,...,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0
559,0759,SsangYong,Korando,2019,CAR_SAFETY,5.0,LHD,2019,1728.0,74.0,...,4.0,4.0,2.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0
560,1149,Audi,A3,2025,CAR_SAFETY,5.0,LHD,2025,1400.0,74.0,...,4.0,4.0,2.0,2.0,4.0,4.0,4.0,4.0,4.0,4.0


In [3]:
df[df['type']=='CAR_SAFETY'].columns.tolist()

['id',
 'make',
 'model',
 'year',
 'type',
 'stars',
 'drive_hand',
 'protocol_year',
 'kerb_weight',
 'safety_assist_score',
 'seat_arrangement',
 'rating_year',
 'adult_occupant_score',
 'child_occupant_score',
 'vru_score',
 'adult_frontal_impact_raw',
 'adult_side_impact_raw',
 'adult_rear_impact_raw',
 'adult_rescue_extrication_raw',
 'child_cop_dynamic_raw',
 'seat1_frontal_airbag',
 'seat3_frontal_airbag',
 'row2_frontal_airbag',
 'seat1_pretensioner',
 'seat3_pretensioner',
 'row2_pretensioner',
 'seat1_load_limiter',
 'seat3_load_limiter',
 'row2_load_limiter',
 'seat1_knee_airbag',
 'seat3_knee_airbag',
 'seat1_side_head_airbag',
 'seat3_side_head_airbag',
 'row2_side_head_airbag',
 'seat1_side_chest_airbag',
 'seat3_side_chest_airbag',
 'row2_side_chest_airbag',
 'seat1_side_pelvis_airbag',
 'seat3_side_pelvis_airbag',
 'row2_side_pelvis_airbag',
 'seat1_centre_lateral_airbag',
 'seat3_centre_lateral_airbag',
 'row2_centre_lateral_airbag',
 'seat1_isofix',
 'seat3_isofix',


In [4]:
df = df[['id','make', 'model','year','type','stars','drive_hand','protocol_year','kerb_weight','seat_arrangement','rating_year', 'seat1_frontal_airbag', 'seat3_frontal_airbag', 'row2_frontal_airbag', 'seat1_pretensioner', 'seat3_pretensioner', 'row2_pretensioner', 'seat1_load_limiter', 'seat3_load_limiter', 'row2_load_limiter', 'seat1_knee_airbag', 'seat3_knee_airbag', 'seat1_side_head_airbag', 'seat3_side_head_airbag', 'row2_side_head_airbag', 'seat1_side_chest_airbag', 'seat3_side_chest_airbag', 'row2_side_chest_airbag', 'seat1_side_pelvis_airbag', 'seat3_side_pelvis_airbag', 'row2_side_pelvis_airbag', 'seat1_centre_lateral_airbag', 'seat3_centre_lateral_airbag', 'row2_centre_lateral_airbag', 'seat1_isofix', 'seat3_isofix', 'row2_isofix', 'row3_isofix', 'seat1_isize', 'seat3_isize', 'row2_isize', 'row3_isize', 'seat1_integrated_child_seat', 'seat3_integrated_child_seat', 'row2_integrated_child_seat', 'row3_integrated_child_seat', 'seat1_airbagcutoffswitch', 'seat3_airbagcutoffswitch', 'row2_airbagcutoffswitch',
 'row3_airbagcutoffswitch', 'seat1_childdetection', 'seat3_childdetection', 'row2_childdetection', 'row3_childdetection', 'has_active_bonnet', 'has_aeb_vru', 'has_cyclistdoorprevention', 'has_aeb_m2w', 'has_aeb_cartocar', 'has_speed_assist',
 'has_lane_assist', 'has_fatigue_detection', 'has_distraction_detection',]]

In [5]:
df

,id,make,model,year,type,stars,drive_hand,protocol_year,kerb_weight,seat_arrangement,...,row3_childdetection,has_active_bonnet,has_aeb_vru,has_cyclistdoorprevention,has_aeb_m2w,has_aeb_cartocar,has_speed_assist,has_lane_assist,has_fatigue_detection,has_distraction_detection
0,v057,IVECO,Daily,2025,VAN_SAFETY,4.0,NaN,2025,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,h019,Scania,R-series,2025,TRUCK_SAFETY,NaN,NaN,2025,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1171,Togg,T10F,2025,CAR_SAFETY,5.0,LHD,2025,2124.0,230,...,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0
3,0850,Toyota,Mirai,2021,CAR_SAFETY,5.0,LHD,2021,1950.0,230,...,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0
4,h023,MAN,TGX,2025,TRUCK_SAFETY,NaN,NaN,2025,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
557,0737,Lexus,ES,2018,CAR_SAFETY,5.0,RHD,2018,1740.0,230,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
558,1021,Volkswagen,Passat,2024,CAR_SAFETY,5.0,LHD,2024,1618.0,230,...,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
559,0759,SsangYong,Korando,2019,CAR_SAFETY,5.0,LHD,2019,1728.0,230,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
560,1149,Audi,A3,2025,CAR_SAFETY,5.0,LHD,2025,1400.0,230,...,0.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0


In [7]:
df = df[df['type']=='CAR_SAFETY']
df

,id,make,model,year,type,stars,drive_hand,protocol_year,kerb_weight,seat_arrangement,...,row3_childdetection,has_active_bonnet,has_aeb_vru,has_cyclistdoorprevention,has_aeb_m2w,has_aeb_cartocar,has_speed_assist,has_lane_assist,has_fatigue_detection,has_distraction_detection
2,1171,Togg,T10F,2025,CAR_SAFETY,5.0,LHD,2025,2124.0,230,...,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0
3,0850,Toyota,Mirai,2021,CAR_SAFETY,5.0,LHD,2021,1950.0,230,...,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0
5,1085,Ford,Capri,2024,CAR_SAFETY,5.0,LHD,2024,2023.0,230,...,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
6,0818,Jeep,Renegade,2019,CAR_SAFETY,3.0,LHD,2019,1484.0,230,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
7,1044,BYD,SEAL,2023,CAR_SAFETY,5.0,LHD,2023,2091.0,230,...,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
557,0737,Lexus,ES,2018,CAR_SAFETY,5.0,RHD,2018,1740.0,230,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
558,1021,Volkswagen,Passat,2024,CAR_SAFETY,5.0,LHD,2024,1618.0,230,...,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
559,0759,SsangYong,Korando,2019,CAR_SAFETY,5.0,LHD,2019,1728.0,230,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
560,1149,Audi,A3,2025,CAR_SAFETY,5.0,LHD,2025,1400.0,230,...,0.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0


In [8]:
df.to_csv('euroncap_cardata_masked.csv', index=False)